<div class="alert alert-success py-2 px-3 mb-2"
     style="font-family: 'Montserrat', sans-serif; font-size: 13px;">

<style>
.alert code {
    background: transparent !important;
    color: inherit !important;
    padding: 0 !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 13px !important;
}
</style>

<strong>Lab 04 – Hub Skills and Integration</strong>

<strong>Notes:</strong> <code>notes/04-hub-skills-and-integration.md</code>

<strong>Goal:</strong> Use the Hub with other tools: search, cache, inference, and pipelines.

<strong>Needs:</strong> <code>HF_READ_TOKEN</code> and <code>HF_WRITE_TOKEN</code> in the repository-root <code>.env</code> file.

<strong>Install:</strong> <code>pip install huggingface_hub transformers torch</code>

<strong>Warning:</strong> The scratch repository and collection created by this lab are deleted in the cleanup cell.

</div>

In [1]:
# ── Lab 04 - Hub Skills and Integration ──
# Notes : notes/04-hub-skills-and-integration.md
# Result: results/04-hub-skills-and-integration.md
# Goal  : use the Hub from other tools (search, cache, inference, pipelines)
# Needs : HF_READ_TOKEN and HF_WRITE_TOKEN in the repo-root .env
# Install: pip install huggingface_hub transformers torch
# Note  : the scratch repo and collection are deleted in the cleanup cell


In [2]:

# ── imports ──
import os
from pathlib import Path
from dotenv import load_dotenv
import huggingface_hub
from huggingface_hub import (
    HfApi, HfFileSystem, InferenceClient, constants, scan_cache_dir,
)

# ── load tokens from .env ──
load_dotenv()
READ_TOKEN = os.environ["HF_READ_TOKEN"]
WRITE_TOKEN = os.environ["HF_WRITE_TOKEN"]
os.environ.pop("HF_TOKEN", None)

# ── one API object, explicit token ──
api = HfApi(token=WRITE_TOKEN)
USER = api.whoami()["name"]

# ── scratch repo used in the discussions cell ──
SCRATCH_REPO = f"{USER}/integration-lab-scratch"

print("huggingface_hub:", huggingface_hub.__version__)
print("user           :", USER)

huggingface_hub: 2.5.0
user           : akshayredekar74


In [4]:
# ── 1. search for models ──
# Top 5 "bert" models by downloads

models = api.list_models(
    search="bert",
    sort="downloads",
    limit=5,
)

top_models = []

for model in models:
    top_models.append(model.id)
    print(model.id, "|", model.downloads, "downloads")

print("\nTop model IDs:")
print(top_models)

google-bert/bert-base-uncased | 33034257 downloads
google-bert/bert-base-cased | 3369275 downloads
google-bert/bert-base-multilingual-uncased | 1912414 downloads
google-bert/bert-base-multilingual-cased | 1798376 downloads
nlpaueb/legal-bert-base-uncased | 1084547 downloads

Top model IDs:
['google-bert/bert-base-uncased', 'google-bert/bert-base-cased', 'google-bert/bert-base-multilingual-uncased', 'google-bert/bert-base-multilingual-cased', 'nlpaueb/legal-bert-base-uncased']


In [5]:
# ── 2. read info about one model ──

MODEL_ID = "google-bert/bert-base-uncased"

info = api.model_info(MODEL_ID)

print("id       :", info.id)
print("task     :", info.pipeline_tag)
print("downloads:", info.downloads)
print("likes    :", info.likes)
print("tags     :", info.tags[:6])

# ── first few files in the repo ──
files = [file.rfilename for file in (info.siblings or [])]

print("files    :", files[:5])

id       : google-bert/bert-base-uncased
task     : fill-mask
downloads: 33034257
likes    : 3094
tags     : ['transformers', 'pytorch', 'tf', 'jax', 'rust', 'coreml']
files    : ['.gitattributes', 'LICENSE', 'README.md', 'config.json', 'coreml/fill-mask/float32_model.mlpackage/Data/com.apple.CoreML/model.mlmodel']


In [6]:
# ── 3. search datasets and Spaces ──

print("datasets:")
for d in api.list_datasets(search="rotten_tomatoes", limit=3):
    print("  ", d.id)

print("spaces:")
for s in api.list_spaces(search="gradio", sort="likes", limit=3):
    print("  ", s.id)

datasets:
   cornell-movie-review-data/rotten_tomatoes
   autoevaluate/autoeval-eval-project-rotten_tomatoes-c34f0918-1313150144
   autoevaluate/autoeval-eval-rotten_tomatoes-default-2a8e03-1477454164
spaces:
   BlinkDL/RWKV-Gradio-2
   BlinkDL/RWKV-Gradio-1
   manavisrani07/gradio-lipsync-wav2lip


In [7]:
# ── 4. HfFileSystem ──
# Treat the Hub like a normal file system

fs = HfFileSystem(token=READ_TOKEN)

files = fs.ls("google-bert/bert-base-uncased", detail=False)
print("files in repo:", len(files))
for f in files[:5]:
    print("  ", f)

# ── read a small file straight from the Hub ──
config_text = fs.read_text("google-bert/bert-base-uncased/config.json")
print(config_text[:120])

files in repo: 14
   google-bert/bert-base-uncased/coreml
   google-bert/bert-base-uncased/.gitattributes
   google-bert/bert-base-uncased/LICENSE
   google-bert/bert-base-uncased/README.md
   google-bert/bert-base-uncased/config.json
{
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "gradient_checkpointing": fal


In [8]:
# ── 5. the local cache ──
# Downloads are stored here so they are not repeated

print("cache folder:", constants.HF_HUB_CACHE)

cache = scan_cache_dir()
print("total size  :", cache.size_on_disk_str)
for repo in list(cache.repos)[:5]:
    print("  ", repo.repo_id, "|", repo.size_on_disk_str)

cache folder: /teamspace/studios/this_studio/.cache/huggingface/hub
total size  : 7.5G
   nvidia/segformer-b5-finetuned-ade-640-640 | 678.3M
   google/flan-t5-base | 993.6M
   Qwen/Qwen2.5-0.5B-Instruct | 11.5M
   google/vit-base-patch16-224 | 346.4M
   facebook/detr-resnet-50 | 166.6M


In [9]:
# ── cache: biggest repos first ──
from huggingface_hub import scan_cache_dir

cache = scan_cache_dir()
print("total:", cache.size_on_disk_str)

for repo in sorted(cache.repos, key=lambda r: r.size_on_disk, reverse=True):
    print(f"{repo.size_on_disk_str:>8} | {repo.repo_type:8} | {repo.repo_id}")

total: 7.5G
    1.6G | model    | facebook/bart-large-cnn
    1.4G | model    | Intel/dpt-large
  993.6M | model    | google/flan-t5-base
  678.3M | model    | nvidia/segformer-b5-finetuned-ade-640-640
  551.0M | model    | gpt2
  497.6M | model    | deepset/roberta-base-squad2
  441.1M | model    | bert-base-uncased
  433.5M | model    | dslim/bert-base-NER
  346.4M | model    | google/vit-base-patch16-224
  268.1M | model    | distilbert-base-uncased-finetuned-sst-2-english
  166.6M | model    | facebook/detr-resnet-50
   83.5M | dataset  | imdb
   11.5M | model    | Qwen/Qwen2.5-0.5B-Instruct
    9.2M | dataset  | hf-internal-testing/librispeech_asr_demo
    7.8M | dataset  | wikitext
    3.4M | model    | HuggingFaceTB/SmolLM2-1.7B-Instruct
    2.7M | model    | roberta-base
    2.3M | model    | meta-llama/Llama-2-7b-hf
    2.2M | model    | t5-base
  888.5K | dataset  | cornell-movie-review-data/rotten_tomatoes
   25.0K | model    | transformers-community/contrastive-search
    7

In [14]:
# ── 6. Inference API ──
from huggingface_hub import InferenceClient

CHAT_MODEL = "Qwen/Qwen3.8-27B"

client = InferenceClient(token=READ_TOKEN)

messages = [
    {"role": "user", "content": "Say hello in five words."}
]

try:
    reply = client.chat_completion(
        messages=messages,
        model=CHAT_MODEL,
        max_tokens=30,
    )

    chat_answer = reply.choices[0].message.content
    print("model says:", chat_answer)

except Exception as e:
    chat_answer = f"failed: {type(e).__name__}: {e}"
    print(chat_answer)

failed: HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6b0897a5-700d7d02155669da0ababd5d;c87640cb-cd1a-4445-a79b-4a0e94d4331c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.


In [ ]:
client = InferenceClient(
    token=READ_TOKEN,
    provider="hf-inference",
)

reply = client.chat_completion(
    model=CHAT_MODEL,
    messages=messages,
    max_tokens=30,
)

In [15]:
# ── 7. transformers pipeline ──
# Downloads a small model, then runs it on your machine

from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    token=READ_TOKEN,
)

result = classifier("I love learning the Hugging Face Hub!")[0]
print(result)

# ── the model is now in the cache ──
print("cache size:", scan_cache_dir().size_on_disk_str)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

{'label': 'POSITIVE', 'score': 0.9995243549346924}
cache size: 7.8G


In [16]:
# ── 7. transformers pipeline ──
# Downloads a small model, then runs it on your machine

from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    token=READ_TOKEN,
)

result = classifier("I love learning the Hugging Face Hub!")[0]
print(result)

# ── the model is now in the cache ──
print("cache size:", scan_cache_dir().size_on_disk_str)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'label': 'POSITIVE', 'score': 0.9995243549346924}
cache size: 7.8G


In [19]:
# ── 8. discussions ──
# Every repo has a discussion tab, like issues on GitHub

# ── make a private scratch repo ──
api.create_repo(SCRATCH_REPO, private=True, exist_ok=True)

# ── open a discussion ──
disc = api.create_discussion(
    repo_id=SCRATCH_REPO,
    title="Question about this model",
    description="Hello! Is this repo ready to use?",
)
print("opened discussion #", disc.num)

# ── list all discussions ──
for d in api.get_repo_discussions(SCRATCH_REPO):
    print("  #", d.num, "|", d.title, "|", d.status)

opened discussion # 1
  # 1 | Question about this model | open


In [17]:
# ── 9. collections ──
# A collection groups models, datasets and Spaces on one page

collection = api.create_collection(
    title="Integration Lab Picks",
    namespace=USER,
    private=True,
)
print("slug:", collection.slug)

# ── add two items ──
api.add_collection_item(collection.slug, item_id="google-bert/bert-base-uncased", item_type="model")
api.add_collection_item(collection.slug, item_id="cornell-movie-review-data/rotten_tomatoes", item_type="dataset")

# ── read it back ──
fresh = api.get_collection(collection.slug)
print("items:", [item.item_id for item in fresh.items])

slug: akshayredekar74/integration-lab-picks-6b0898206f68f5056d6d4ff0
items: ['google-bert/bert-base-uncased', 'cornell-movie-review-data/rotten_tomatoes']


In [20]:
# ── 10. save results ──
# Writes results/04-hub-skills-and-integration.md

results_dir = Path.cwd().parent / "results"
results_dir.mkdir(exist_ok=True)

text = f"""# 04 - Hub Skills and Integration

huggingface_hub {huggingface_hub.__version__}

| Skill | Result |
|---|---|
| Search models | {top_models[:3]} |
| Model info | {info.id}, task: {info.pipeline_tag} |
| File system | {len(files)} files listed |
| Local cache | {scan_cache_dir().size_on_disk_str} |
| Inference API | {chat_answer} |
| Pipeline | {result["label"]} ({result["score"]:.2f}) |
| Collection items | {[item.item_id for item in fresh.items]} |
"""

(results_dir / "04-hub-skills-and-integration.md").write_text(text)
print("saved:", results_dir / "04-hub-skills-and-integration.md")

saved: /teamspace/studios/this_studio/All-about-finetuning/finetuning/00_platform/results/04-hub-skills-and-integration.md


In [21]:
# # ── 11. clean up ──
# # Deletes the scratch repo and the collection. Run this last.

# api.delete_repo(SCRATCH_REPO)
# api.delete_collection(collection.slug)
# print("deleted scratch repo and collection")

In [22]:
# ── cache: delete everything ──
# Frees all 7.5 GB. Models are downloaded again when you next use them.

import shutil
from pathlib import Path
from huggingface_hub import constants

shutil.rmtree(constants.HF_HUB_CACHE, ignore_errors=True)
print("cache folder exists:", Path(constants.HF_HUB_CACHE).exists())   # expect: False

cache folder exists: False
